In [28]:
import gymnasium as gym
import numpy as np
from collections import defaultdict
from RL_Algorithms import RL_Algorithm
from Updaters import *
from Strategies import *
from State_Representations import *


In [29]:
env = gym.make("Pendulum-v1")

obs_cardinality = (12, 12, 12)  # cos(theta), sin(theta), theta_dot
obs_space_high = np.array([1.0, 1.0, 8.0])
obs_space_low = np.array([-1.0, -1.0, -8.0])

n_actions = 9
action_space = np.linspace(-2, 2, n_actions)

episodes = 100

In [30]:

class classic_Q_Learning(RL_Algorithm):
    def __init__(self, ID, alpha : float, gamma : float, eps_decay : float, eps_min : float):
        super().__init__()
        self.strategy = Epsilon_Decay(epsilon = 1.0, epsilon_decay = eps_min, epsilon_min = eps_min)
        self.updater = Q_Learning(alpha=alpha, gamma=gamma)

class classic_SARSA(RL_Algorithm):
    def __init__(self, ID, alpha : float, gamma : float, eps_decay : float, eps_min : float):
        super().__init__()
        self.strategy = Epsilon_Decay(epsilon = 1.0, epsilon_decay = eps_min, epsilon_min = eps_min)
        self.updater = SARSA(alpha=alpha, gamma=gamma)

In [31]:
gammas = [0.99, 0.9]
alphas = [0.1, 0.01]
eps_decays = [0.9995, 0.9999]
eps_mins = [0.05]

algoritms = {}

In [32]:
i = 0
for gamma in gammas:
    for alpha in alphas:
        for eps_decay in eps_decays:
            for eps_min in eps_mins:
                q = classic_Q_Learning(ID=i, alpha=alpha, gamma=gamma, eps_decay=eps_decay, eps_min=eps_min)
                algoritms[i] = q
                i += 1
                s = classic_SARSA(ID=i, alpha=alpha, gamma=gamma, eps_decay=eps_decay, eps_min=eps_min)
                algoritms[i] = s
                i += 1

In [33]:
algorithm_rewards = defaultdict(list)

alg : RL_Algorithm
for ix, (id, alg) in enumerate(algoritms.items()):
    q_table = Q_Table(obs_cardinality, obs_space_low, obs_space_high, (n_actions, ))
    alg.Set_State_Representation(q_table)

    for episode in range(episodes):
        obs, _ = env.reset()
        state = q_table.Observation_To_State(obs)

        total_reward = 0
        done = False

        while not done:

            action_index, next_state, reward, terminated, truncated = alg.Step(action_space, state, env)

            state = next_state
            total_reward += reward

            done = terminated or truncated

        alg.Episode_Ended()
        algorithm_rewards[id].append(total_reward)

        if (episode + 1) % 5000 == 0:
            print(f"Epizod {episode+1}, total reward: {total_reward:.2f}, epsilon: {alg.Get_Strategy().Get_Epsilon():.2f}")


    print(f"Trening zakończony! Total reward: {total_reward:.2f}")

Trening zakończony! Total reward: -1249.99
Trening zakończony! Total reward: -848.48
Trening zakończony! Total reward: -1288.95
Trening zakończony! Total reward: -1283.36
Trening zakończony! Total reward: -984.94
Trening zakończony! Total reward: -1060.58
Trening zakończony! Total reward: -896.65
Trening zakończony! Total reward: -1036.29
Trening zakończony! Total reward: -975.76
Trening zakończony! Total reward: -1330.09
Trening zakończony! Total reward: -1293.54
Trening zakończony! Total reward: -1699.05
Trening zakończony! Total reward: -1508.35
Trening zakończony! Total reward: -1278.62
Trening zakończony! Total reward: -882.04
Trening zakończony! Total reward: -1738.07


In [41]:
final_rewards : dict[int, float] = {}
for ix, (id, rewards) in enumerate(algorithm_rewards.items()):
    final_rewards[id] = rewards[-1].item()

final_rewards = dict(sorted(final_rewards.items(), key=lambda item: -item[1]))

print(final_rewards)

{1: -848.4832434938342, 14: -882.0446721666397, 6: -896.6470163045667, 8: -975.7553899324876, 4: -984.9370418948547, 7: -1036.287772179898, 5: -1060.5828917029348, 0: -1249.9905303588025, 13: -1278.6221411529157, 3: -1283.3581232458835, 2: -1288.9460825980577, 10: -1293.5415703271983, 9: -1330.0885603227123, 12: -1508.3500550681918, 11: -1699.0488626358708, 15: -1738.0742865705465}


In [35]:

env = gym.make("Pendulum-v1", render_mode = "human")

obs, _ = env.reset()
state =  list(algoritms.values())[-1].Get_State_Representation().Observation_To_State(obs)
strategy = Greedy()
eval_rew = 0

for _ in range(200):
    action_idx : int = strategy.Get_Action_Index(q_table, state)
    action : np.array = np.array(action_space[action_idx])

    obs, rew, terminated, truncated, _ = env.step([action.item()])
    eval_rew += rew
    state = q_table.Observation_To_State(obs)

print(f"Eval rew: {eval_rew}")
env.close()

Eval rew: -1020.3924086290314
